In [ ]:
import os
import sys
ROOT_DIR = '/root/Desktop/data/private/DIG-dig-stable'
sys.path.append(ROOT_DIR)
sys.path.insert(0, ROOT_DIR)
import torch
import h5py
from models.model_set_mil import MIL_Attention_FC_surv
from scipy import ndimage
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgb


TISSUE_NAMES = [
    "Tumor", "Empty", "Fibrous", "Inflammation",
    "Necrosis", "Normal", "Reactive", "Steatosis"
]

TISSUE_COLORS = [
    "#E64B35", "#FFFFFF", "#7E57C2", "#6E6E6E",
    "#F06292", "#66BB6A", "#26A69A", "#F39C12"
]

In [ ]:
def load_segmap(segmap_npy):
    seg_prob = np.load(segmap_npy)
    seg_label = np.argmax(seg_prob, axis=2)
    return seg_prob, seg_label

def load_coords_from_h5(h5_path, coord_key="coords"):
    with h5py.File(h5_path, "r") as f:
        coords = f[coord_key][:]
    return coords.astype(np.int32)

def get_foreground_bbox_clean(seg_label, empty_label=1, min_area=500):
    fg_mask = seg_label != empty_label

    labeled, num = ndimage.label(fg_mask)
    if num == 0:
        raise ValueError("Foreground mask is empty.")

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0

    keep_labels = np.where(sizes >= min_area)[0]

    if len(keep_labels) == 0:
        fg_clean = fg_mask
    else:
        fg_clean = np.isin(labeled, keep_labels)

    ys, xs = np.where(fg_clean)
    if len(xs) == 0 or len(ys) == 0:
        raise ValueError("Clean foreground mask is empty.")

    return xs.min(), xs.max(), ys.min(), ys.max(), fg_clean



def seg_label_to_rgb(seg_label):
    rgb = np.zeros((*seg_label.shape, 3))

    for i, color in enumerate(TISSUE_COLORS):
        rgb[seg_label == i] = to_rgb(color)

    return rgb

In [ ]:
def find_existing_file_by_patient_id(patient_id, search_dir, suffix):
    candidates = []
    for f in os.listdir(search_dir):
        if not f.endswith(suffix):
            continue
        if patient_id in f:
            candidates.append(f)
    if len(candidates) == 0:
        return None
    if len(candidates) > 1:
        print(f"[Warning] Multiple matches for {patient_id}")
        print(candidates)
    candidates = sorted(candidates)

    return os.path.join(search_dir, candidates[0])


def load_h5_features_and_coords(h5_path):
    with h5py.File(h5_path, "r") as f:
        print(f"[H5 keys] {h5_path}: {list(f.keys())}")

        if "features" in f:
            feats = f["features"][:]
        elif "feats" in f:
            feats = f["feats"][:]
        else:
            raise KeyError(f"No features/feats key found in {h5_path}")

        if "coords" in f:
            coords = f["coords"][:]
        else:
            coords = None

    return feats, coords


@torch.no_grad()
def get_amil_attention_for_h5(model, h5_path, device):
    feats, coords = load_h5_features_and_coords(h5_path)

    x_path = torch.from_numpy(feats).float().to(device)

    model.eval()
    out = model(x_path=x_path)

    A = out["A"]

    if isinstance(A, torch.Tensor):
        A = A.detach().cpu().numpy()

    A = np.asarray(A).reshape(-1)

    if len(A) != feats.shape[0]:
        raise ValueError(f"Attention length {len(A)} != feature number {feats.shape[0]} "f"for {h5_path}")

    return {
        "attention": A,
        "features": feats,
        "coords": coords,
        "h5_path": h5_path,
        "num_patches": feats.shape[0],
    }


def build_amil_sample_items_from_split(
        splits_csv,
        h5_dir,
        segmap_dir,
        model,
        device,
        split_cols=("train", "validation"),
        seg_suffix=".npy",
        h5_suffix=".h5",
        overlay_save_dir=None,
):
    split_df = pd.read_csv(splits_csv)

    sample_ids = []
    for col in split_cols:
        if col not in split_df.columns:
            raise ValueError(f"Column {col} not found in split csv.")

        sample_ids.extend(split_df[col].dropna().tolist())

    sample_ids = [str(x) for x in sample_ids]

    if overlay_save_dir is not None:
        os.makedirs(overlay_save_dir, exist_ok=True)

    sample_items = []
    missing_h5 = []
    missing_seg = []
    failed_attention = []

    for idx, patient_id in enumerate(sample_ids):
        print(f"\n[{idx+1}/{len(sample_ids)}] Processing {patient_id}")

        h5_path = find_existing_file_by_patient_id(patient_id=patient_id, search_dir=h5_dir, suffix=h5_suffix)
        segmap_npy = find_existing_file_by_patient_id(patient_id=patient_id, search_dir=segmap_dir, suffix=seg_suffix)

        if h5_path is None:
            print(f"[Missing h5] {patient_id}")
            missing_h5.append(patient_id)
            continue

        if segmap_npy is None:
            print(f"[Missing segmap] {patient_id}")
            missing_seg.append(patient_id)
            continue

        try:
            attn_info = get_amil_attention_for_h5(model=model, h5_path=h5_path, device=device)
        except Exception as e:
            print(f"[Failed attention] {patient_id}: {e}")
            failed_attention.append((patient_id, str(e)))
            continue

        if overlay_save_dir is not None:
            save_overlay_path = os.path.join(overlay_save_dir, f"{patient_id}_amil_top_attention_overlay.png")
        else:
            save_overlay_path = None

        item = {
            "patient_id": patient_id,
            "sample_idx": idx,
            "attention": attn_info["attention"],
            "h5_path": h5_path,
            "segmap_npy": segmap_npy,
            "save_overlay_path": save_overlay_path,
            "num_patches": attn_info["num_patches"],
        }

        sample_items.append(item)

    print("\n=== Summary ===")
    print("Valid samples:", len(sample_items))
    print("Missing h5:", len(missing_h5))
    print("Missing segmap:", len(missing_seg))
    print("Failed attention:", len(failed_attention))

    return sample_items, {
        "missing_h5": missing_h5,
        "missing_segmap": missing_seg,
        "failed_attention": failed_attention,
    }

In [ ]:
def coords_to_seg_boxes_bbox_sample_scale(coords, seg_label, patch_size=512, empty_label=1):

    coords = coords.astype(np.float64)

    # WSI / CLAM coords bbox
    x_min_wsi = coords[:, 0].min()
    y_min_wsi = coords[:, 1].min()
    x_max_wsi = coords[:, 0].max()
    y_max_wsi = coords[:, 1].max()

    wsi_w = (x_max_wsi - x_min_wsi) + patch_size
    wsi_h = (y_max_wsi - y_min_wsi) + patch_size

    x_min_seg, x_max_seg, y_min_seg, y_max_seg, fg_mask = get_foreground_bbox_clean(
        seg_label,
        empty_label=empty_label,
        min_area=500
    )

    seg_w = x_max_seg - x_min_seg + 1
    seg_h = y_max_seg - y_min_seg + 1

    scale_x = seg_w / wsi_w
    scale_y = seg_h / wsi_h

    x0 = coords[:, 0] - x_min_wsi
    y0 = coords[:, 1] - y_min_wsi

    x1 = x_min_seg + x0 * scale_x
    y1 = y_min_seg + y0 * scale_y
    x2 = x_min_seg + (x0 + patch_size) * scale_x
    y2 = y_min_seg + (y0 + patch_size) * scale_y

    ix = ((x1 + x2) / 2).astype(int)
    iy = ((y1 + y2) / 2).astype(int)

    Hm, Wm = seg_label.shape
    ix = np.clip(ix, 0, Wm - 1)
    iy = np.clip(iy, 0, Hm - 1)

    mapping_info = {
        "x_min_wsi": float(x_min_wsi),
        "y_min_wsi": float(y_min_wsi),
        "x_max_wsi": float(x_max_wsi),
        "y_max_wsi": float(y_max_wsi),
        "wsi_w": float(wsi_w),
        "wsi_h": float(wsi_h),
        "x_min_seg": int(x_min_seg),
        "x_max_seg": int(x_max_seg),
        "y_min_seg": int(y_min_seg),
        "y_max_seg": int(y_max_seg),
        "seg_w": int(seg_w),
        "seg_h": int(seg_h),
        "scale_x": float(scale_x),
        "scale_y": float(scale_y),
        "scale_ratio_x_over_y": float(scale_x / scale_y),
    }

    return iy, ix, x1, y1, x2, y2, fg_mask, mapping_info

def pool_patch_tissue_from_boxes_foreground_only(
    seg_prob,
    x1, y1, x2, y2,
    empty_label=1,
    min_fg_ratio=0.05
):
    Hm, Wm, C = seg_prob.shape
    N = len(x1)

    patch_tissue_prob = np.zeros((N, C), dtype=np.float32)
    patch_tissue_label = np.full(N, empty_label, dtype=np.int64)
    patch_fg_ratio = np.zeros(N, dtype=np.float32)

    seg_label = np.argmax(seg_prob, axis=2)

    for i in range(N):
        xa = int(np.floor(x1[i]))
        ya = int(np.floor(y1[i]))
        xb = int(np.ceil(x2[i]))
        yb = int(np.ceil(y2[i]))

        xa = np.clip(xa, 0, Wm - 1)
        xb = np.clip(xb, xa + 1, Wm)
        ya = np.clip(ya, 0, Hm - 1)
        yb = np.clip(yb, ya + 1, Hm)

        region_prob = seg_prob[ya:yb, xa:xb, :]
        region_label = seg_label[ya:yb, xa:xb]

        if region_prob.size == 0:
            continue

        fg_mask = region_label != empty_label
        patch_fg_ratio[i] = fg_mask.mean()

        if patch_fg_ratio[i] < min_fg_ratio:
            patch_tissue_prob[i, empty_label] = 1.0
            patch_tissue_label[i] = empty_label
            continue

        fg_prob = region_prob[fg_mask]
        mean_prob = fg_prob.mean(axis=0)

        mean_prob[empty_label] = 0.0
        s = mean_prob.sum()
        if s > 0:
            mean_prob = mean_prob / s

        patch_tissue_prob[i] = mean_prob
        patch_tissue_label[i] = int(np.argmax(mean_prob))

    return patch_tissue_prob, patch_tissue_label, patch_fg_ratio

In [ ]:
def adaptive_topk_attention(attn, min_k=20, max_k=300, top_frac=0.05):
    attn = np.asarray(attn).reshape(-1)
    n = len(attn)

    k = int(np.ceil(n * top_frac))
    k = max(min_k, k)
    k = min(max_k, k)
    k = min(k, n)

    top_idx = np.argsort(attn)[-k:][::-1]
    return top_idx, k


def summarize_segmap_tissue_pixels(seg_label, tissue_names=TISSUE_NAMES, ignore_empty=True, empty_label=1):
    seg_label = np.asarray(seg_label)

    if ignore_empty:
        valid = seg_label != empty_label
    else:
        valid = np.ones_like(seg_label, dtype=bool)

    labels = seg_label[valid]

    counts = np.bincount(labels.ravel(), minlength=len(tissue_names))

    total = counts.sum()
    ratios = counts / total if total > 0 else np.full(len(tissue_names), np.nan)

    summary = {"wsi_total_tissue_pixels": int(total)}

    for i, name in enumerate(tissue_names):
        summary[f"wsi_{name}_pixel_count"] = int(counts[i])
        summary[f"wsi_{name}_pixel_ratio"] = float(ratios[i])

    return summary


def summarize_topk_patch_tissue_pixels(
        seg_label,
        top_patch_idx,
        x1, y1, x2, y2,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
):
    counts = np.zeros(len(tissue_names), dtype=np.int64)

    for idx in top_patch_idx:
        idx = int(idx)

        patch_label = seg_label[
            int(y1[idx]):int(y2[idx]),
            int(x1[idx]):int(x2[idx])
        ]

        if patch_label.size == 0:
            continue

        if ignore_empty:
            patch_label = patch_label[patch_label != empty_label]

        if patch_label.size == 0:
            continue

        counts += np.bincount(patch_label.ravel(), minlength=len(tissue_names))

    total = counts.sum()
    ratios = counts / total if total > 0 else np.full(len(tissue_names), np.nan)

    summary = {"topk_total_tissue_pixels": int(total)}

    for i, name in enumerate(tissue_names):
        summary[f"topk_{name}_pixel_count"] = int(counts[i])
        summary[f"topk_{name}_pixel_ratio"] = float(ratios[i])

    return summary


def summarize_topk_patch_tissue_labels(
        patch_tissue_label,
        patch_tissue_prob,
        top_patch_idx,
        tissue_names=TISSUE_NAMES,
):
    labels = patch_tissue_label[top_patch_idx]
    probs = patch_tissue_prob[top_patch_idx]

    counts = np.bincount(labels, minlength=len(tissue_names))

    total = counts.sum()
    ratios = counts / total if total > 0 else np.full(len(tissue_names), np.nan)
    mean_prob = probs.mean(axis=0) if len(probs) > 0 else np.full(len(tissue_names), np.nan)

    summary = {"topk_num_patches": int(total)}

    for i, name in enumerate(tissue_names):
        summary[f"topk_{name}_patch_count"] = int(counts[i])
        summary[f"topk_{name}_patch_ratio"] = float(ratios[i])
        summary[f"topk_{name}_mean_prob"] = float(mean_prob[i])

    return summary


def plot_amil_attention_topk_overlay(
        seg_label,
        ix,
        iy,
        top_patch_idx,
        save_path=None,
        title="AMIL top-attention patches over tissue segmentation",
):
    plt.figure(figsize=(8, 8))

    seg_rgb = seg_label_to_rgb(seg_label)
    plt.imshow(seg_rgb)

    plt.scatter(
        ix[top_patch_idx],
        iy[top_patch_idx],
        s=40,
        c="white",
        alpha=0.6,
        linewidths=0,
        zorder=3
    )

    plt.scatter(
        ix[top_patch_idx],
        iy[top_patch_idx],
        s=15,
        c="#00BCD4",
        edgecolors="#006064",
        linewidths=0.8,
        zorder=4
    )

    patches = [
        mpatches.Patch(
            facecolor=TISSUE_COLORS[i],
            edgecolor="#333333",
            linewidth=0.6,
            label=f"{i}: {TISSUE_NAMES[i]}"
        )
        for i in range(len(TISSUE_NAMES))
    ]

    patches.append(plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#00BCD4", markeredgecolor="#006064", markersize=8, linestyle="None", label="AMIL top-attention patches"))

    plt.legend(
        handles=patches,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False
    )

    plt.title(title)
    plt.axis("off")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"[Saved] {save_path}")

    plt.show()
    plt.close()


def overlay_amil_attention_with_pathfinder_refined(
        attention,
        segmap_npy,
        h5_path,
        patch_size=1024,
        save_overlay_path=None,
        patient_id=None,
        sample_idx=None,
        top_frac=0.05,
        min_k=20,
        max_k=300,
):

    # 1) segmap
    seg_prob, seg_label = load_segmap(segmap_npy)

    # 2) coords
    coords = load_coords_from_h5(h5_path)

    attention = np.asarray(attention).reshape(-1)

    # 3) mapping
    iy, ix, x1, y1, x2, y2, fg_mask, mapping_info = coords_to_seg_boxes_bbox_sample_scale(
        coords=coords,
        seg_label=seg_label,
        patch_size=patch_size,
        empty_label=1
    )

    print(
        f"Sample-scale mapping: "
        f"scale_x={mapping_info['scale_x']:.6f}, "
        f"scale_y={mapping_info['scale_y']:.6f}, "
        f"scale_ratio={mapping_info['scale_ratio_x_over_y']:.4f}"
    )

    # 4) patch tissue category
    patch_tissue_prob, patch_tissue_label, patch_fg_ratio = pool_patch_tissue_from_boxes_foreground_only(
        seg_prob=seg_prob,
        x1=x1, y1=y1, x2=x2, y2=y2,
        empty_label=1,
        min_fg_ratio=0.05
    )

    # 5) adaptive top-k attention patches
    top_patch_idx, k = adaptive_topk_attention(
        attention,
        min_k=min_k,
        max_k=max_k,
        top_frac=top_frac,
    )

    print(f"Number of patches: {len(attention)}")
    print(f"Adaptive top-k: {k}")
    print("Top-k mean foreground ratio:", patch_fg_ratio[top_patch_idx].mean())
    print("Top-k low-foreground patches:", (patch_fg_ratio[top_patch_idx] < 0.05).sum(),"/", len(top_patch_idx))

    # 6) top-k patch-level tissue ratio
    topk_patch_summary = summarize_topk_patch_tissue_labels(
        patch_tissue_label=patch_tissue_label,
        patch_tissue_prob=patch_tissue_prob,
        top_patch_idx=top_patch_idx,
        tissue_names=TISSUE_NAMES,
    )

    # 7) top-k pixel-level tissue ratio
    topk_pixel_summary = summarize_topk_patch_tissue_pixels(
        seg_label=seg_label,
        top_patch_idx=top_patch_idx,
        x1=x1, y1=y1, x2=x2, y2=y2,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
    )

    # 8) segmap pixel-level tissue ratio
    wsi_pixel_summary = summarize_segmap_tissue_pixels(
        seg_label=seg_label,
        tissue_names=TISSUE_NAMES,
        ignore_empty=True,
        empty_label=1,
    )

    # 9) overlay
    plot_amil_attention_topk_overlay(
        seg_label=seg_label,
        ix=ix,
        iy=iy,
        top_patch_idx=top_patch_idx,
        save_path=save_overlay_path,
        title="AMIL top-attention patches over tissue segmentation",
    )

    return {
        "patient_id": patient_id,
        "sample_idx": sample_idx,
        "attention": attention,
        "top_patch_idx": top_patch_idx,
        "top_k": k,
        "coords": coords,
        "seg_prob": seg_prob,
        "seg_label": seg_label,
        "ix": ix,
        "iy": iy,
        "x1": x1,
        "y1": y1,
        "x2": x2,
        "y2": y2,
        "patch_tissue_label": patch_tissue_label,
        "patch_tissue_prob": patch_tissue_prob,
        "patch_fg_ratio": patch_fg_ratio,
        "topk_patch_summary": topk_patch_summary,
        "topk_pixel_summary": topk_pixel_summary,
        "wsi_pixel_summary": wsi_pixel_summary,
        "mapping_info": mapping_info,
    }

In [ ]:
def save_amil_attention_tissue_summary_csv(all_results, save_csv_path):
    rows = []

    for res in all_results:
        row = {
            "patient_id": res.get("patient_id", None),
            "sample_idx": res.get("sample_idx", None),
            "top_k": res.get("top_k", None),
            "num_patches": len(res.get("attention", [])),
        }

        topk_patch_summary = res.get("topk_patch_summary", {})
        for k, v in topk_patch_summary.items():
            row[k] = v

        topk_pixel_summary = res.get("topk_pixel_summary", {})
        for k, v in topk_pixel_summary.items():
            row[k] = v

        wsi_pixel_summary = res.get("wsi_pixel_summary", {})
        for k, v in wsi_pixel_summary.items():
            row[k] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(save_csv_path, index=False)

    print(f"[Saved CSV] {save_csv_path}")
    print(df.head())

    return df

In [ ]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

model = MIL_Attention_FC_surv(size_arg="small", dropout=0.25, n_classes=4)
ckpt_path = '/root/Desktop/data/private/hjx_product/results_final_0614/5foldcv/AMIL_nll_surv_a0.0_5foldcv_gc32/lihc_AMIL_nll_surv_a0.0_5foldcv_gc32_s42/s_2_minloss_checkpoint.pt'
state_dict = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

splits_dir = "/root/Desktop/data/private/LIHC/5foldcv/lihc_343/splits_2.csv"
segmap_dir = "/root/Desktop/data/private/PagSegNet/seg_hjx/resnext50_32x4d/"
h5_dir = "/root/Desktop/data/private/LIHC/feature_DX_UNI2/h5_files/"

overlay_save_dir = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/AMIL/vis_overlay"
save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/AMIL/train_tissue_summary.csv"

sample_items, info = build_amil_sample_items_from_split(
    splits_csv=splits_dir,
    h5_dir=h5_dir,
    segmap_dir=segmap_dir,
    model=model,
    device=device,
    split_cols=("train", "validation"),
    overlay_save_dir=overlay_save_dir,
)

all_amil_results = []

for item in sample_items:
    save_path = os.path.join(overlay_save_dir,f"{item['patient_id']}_AMIL_overlay.png")

    res = overlay_amil_attention_with_pathfinder_refined(
        attention=item["attention"],
        segmap_npy=item["segmap_npy"],
        h5_path=item["h5_path"],
        patch_size=1024,
        save_overlay_path=item["save_overlay_path"],
        patient_id=item["patient_id"],
        sample_idx=item["sample_idx"],
        top_frac=0.08,
        min_k=30,
        max_k=360,
    )

    all_amil_results.append(res)

amil_df = save_amil_attention_tissue_summary_csv(all_amil_results, save_csv_path=save_csv_path)

In [ ]:
save_csv_path = "/root/Desktop/data/private/hjx_product/results_pgexplainer_0614/AMIL/test_tissue_summary.csv"

sample_items, info = build_amil_sample_items_from_split(
    splits_csv=splits_dir,
    h5_dir=h5_dir,
    segmap_dir=segmap_dir,
    model=model,
    device=device,
    split_cols=("test",),
    overlay_save_dir=overlay_save_dir,
)

all_amil_results = []

for item in sample_items:
    save_path = os.path.join(overlay_save_dir,f"{item['patient_id']}_AMIL_overlay.png")

    res = overlay_amil_attention_with_pathfinder_refined(
        attention=item["attention"],
        segmap_npy=item["segmap_npy"],
        h5_path=item["h5_path"],
        patch_size=1024,
        save_overlay_path=item["save_overlay_path"],
        patient_id=item["patient_id"],
        sample_idx=item["sample_idx"],
        top_frac=0.08,
        min_k=30,
        max_k=360,
    )

    all_amil_results.append(res)

amil_df = save_amil_attention_tissue_summary_csv(all_amil_results, save_csv_path=save_csv_path)